In [2]:
pip install -r requerimiento.txt --quiet

Note: you may need to restart the kernel to use updated packages.


## EDA Presupuesto de Egresos de la Federación

In [1]:
import warnings
import sys
import pandas as pd
import requests
import bs4
import openpyxl
from io import BytesIO
import sqlite3
import os
pd.set_option("display.max_colwidth", None) 

In [2]:
base_url = "https://repodatos.atdt.gob.mx/api_update/secretaria_hacienda/presupuesto_egresos_federacion_avance_gasto_trimestre/PEF_avance_gasto.csv"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(base_url, headers=headers)
if response.status_code == 200:
    df = pd.read_csv(BytesIO(response.content))
else:
    print("Error:", response.status_code)

C:\Users\david\AppData\Local\Temp\ipykernel_16488\1201489562.py:5: DtypeWarning: Columns (0: id_clave_cartera) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(BytesIO(response.content))


In [ ]:
# primero vamos a ver las columnas que tenemos y la información general del dataframe
df.head()

,ciclo,id_ramo,desc_ramo,id_ur,desc_ur,gpo_funcional,desc_gpo_funcional,id_funcion,desc_funcion,id_subfuncion,...,desc_ff,id_entidad_federativa,entidad_federativa,id_clave_cartera,monto_aprobado,monto_modificado,monto_aprobado_mensual,monto_modificado_mensual,monto_pagado,tipo_gasto
0,2026.0,50.0,Instituto Mexicano del Seguro Social,GYR,Instituto Mexicano del Seguro Social,1.0,Gobierno,3.0,Coordinación de la Política de Gobierno,4.0,...,Ingresos Propios,1.0,Aguascalientes,0,702742.0,706586.0,173199.0,177043.0,184784.0,PROGRAMABLE
1,2026.0,50.0,Instituto Mexicano del Seguro Social,GYR,Instituto Mexicano del Seguro Social,1.0,Gobierno,3.0,Coordinación de la Política de Gobierno,4.0,...,Ingresos Propios,2.0,Baja California,0,2017800.0,2028482.0,496320.0,507002.0,527376.9,PROGRAMABLE
2,2026.0,50.0,Instituto Mexicano del Seguro Social,GYR,Instituto Mexicano del Seguro Social,1.0,Gobierno,3.0,Coordinación de la Política de Gobierno,4.0,...,Ingresos Propios,3.0,Baja California Sur,0,713555.0,711743.0,175279.0,173467.0,167008.0,PROGRAMABLE
3,2026.0,50.0,Instituto Mexicano del Seguro Social,GYR,Instituto Mexicano del Seguro Social,1.0,Gobierno,3.0,Coordinación de la Política de Gobierno,4.0,...,Ingresos Propios,5.0,Coahuila,0,1543644.0,1526149.0,379379.0,361884.0,322406.8,PROGRAMABLE
4,2026.0,50.0,Instituto Mexicano del Seguro Social,GYR,Instituto Mexicano del Seguro Social,1.0,Gobierno,3.0,Coordinación de la Política de Gobierno,4.0,...,Ingresos Propios,6.0,Colima,0,646295.0,649814.0,159239.0,162758.0,122467.7,PROGRAMABLE


In [ ]:
# vamos a ver con que columnas contamos para hacer el análisis
print(df.columns)
print(df.shape)

Index(['ciclo', 'id_ramo', 'desc_ramo', 'id_ur', 'desc_ur', 'gpo_funcional',
       'desc_gpo_funcional', 'id_funcion', 'desc_funcion', 'id_subfuncion',
       'desc_subfuncion', 'id_ai', 'desc_ai', 'id_modalidad', 'desc_modalidad',
       'id_pp', 'desc_pp', 'id_capitulo', 'desc_capitulo', 'id_concepto',
       'desc_concepto', 'id_partida_generica', 'desc_partida_generica',
       'id_partida_especifica', 'desc_partida_especifica', 'id_tipogasto',
       'desc_tipogasto', 'id_ff', 'desc_ff', 'id_entidad_federativa',
       'entidad_federativa', 'id_clave_cartera', 'monto_aprobado',
       'monto_modificado', 'monto_aprobado_mensual',
       'monto_modificado_mensual', 'monto_pagado', 'tipo_gasto'],
      dtype='str')
(167593, 38)


Bien, vamos a traer los metadatos de la siguiente [liga](https://www.transparenciapresupuestaria.gob.mx/Datos-Abiertos) en el Diccionario de [términos presupuestarios](https://www.transparenciapresupuestaria.gob.mx/work/models/PTP/DatosAbiertos/Diccionarios/diccionario_presupuesto.xlsx). 

In [46]:
url = "https://www.transparenciapresupuestaria.gob.mx/work/models/PTP/DatosAbiertos/Diccionarios/diccionario_presupuesto.xlsx"
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers = headers, verify=False)
if response.status_code == 200:
    df_diccionario = pd.read_excel(BytesIO(response.content), engine='openpyxl',skiprows=4)
else:
    print(f"Error al descargar el archivo: {response.status_code}")

C:\Users\david\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.transparenciapresupuestaria.gob.mx'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [47]:
df_diccionario.head()

,Variable,Descripción Variable,Etiqueta de Datos Abiertos
0,Actividad Institucional,"Clave que identifica las acciones sustantivas o de apoyo que realizan los ejecutores de gasto con el fin de dar cumplimiento a los objetivos y metas contenidos en los programas, de conformidad con las atribuciones que les señala su respectiva ley orgánica o el ordenamiento jurídico que les es aplicable (3 dígitos).\nEsta información está disponible de 2008 a 2018.",ID_AI
1,Adefas,"Los adeudos de ejercicios fiscales anteriores (ADEFAS) son los gastos devengados y no pagados al último día de cada ejercicio fiscal, derivados del ejercicio del Presupuesto de Egresos, incluido el gasto devengado de las dependencias cuya cuenta por liquidar correspondiente está pendiente de presentarse a la Tesorería, así como las cuentas por liquidar presentadas a ésta que quedaron pendientes de pago.\nEsta información está disponible de 2014 a 2016.",MONTO_ADEFAS
2,Aprobado,"Son las asignaciones presupuestarias anuales comprendidas en el Presupuesto de Egresos a nivel de clave presupuestaria en el caso de los ramos autónomos, administrativos y generales, y a nivel de los rubros de gasto que aparecen en las carátulas de flujo de efectivo para las entidades.\nEsta información está disponible de 2008 a 2018.",MONTO_APROBADO
3,Capítulo,"Corresponde a la clave para el mayor nivel de agregación del Clasificador por Objeto del Gasto, que sirve para identificar conjuntos homogéneos y ordenados de bienes y servicios requeridos por las Entidades y Dependencias de todos los niveles de gobierno. En la clave del Clasificador por Objeto del Gasto, puede identificarse con el primer dígito de izquierda a derecha.\nEsta información está disponible de 2008 a 2018.",ID_CAPITULO
4,Ciclo,Determina el ciclo presupuestario de referencia.\nEsta información está disponible de 2008 a 2018.,CICLO


El diccionario nos ayudo a entender la anaturaleza de la base de datos. La mayor parte son ID, por lo que conviene hacer un modelo estrella en SQL. Pero antes vamos a checar la calidad de datos y la granularidad.

In [48]:
nulos = df.isnull().sum()
nulos_mayores_cero = nulos[nulos > 0]    
print(nulos_mayores_cero)   
print(df.info())

Series([], dtype: int64)
<class 'pandas.DataFrame'>
RangeIndex: 167593 entries, 0 to 167592
Data columns (total 38 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ciclo                     167593 non-null  float64
 1   id_ramo                   167593 non-null  float64
 2   desc_ramo                 167593 non-null  str    
 3   id_ur                     167593 non-null  str    
 4   desc_ur                   167593 non-null  str    
 5   gpo_funcional             167593 non-null  float64
 6   desc_gpo_funcional        167593 non-null  str    
 7   id_funcion                167593 non-null  float64
 8   desc_funcion              167593 non-null  str    
 9   id_subfuncion             167593 non-null  float64
 10  desc_subfuncion           167593 non-null  str    
 11  id_ai                     167593 non-null  float64
 12  desc_ai                   167593 non-null  str    
 13  id_modalidad              1675

In [49]:
niveles = ['id_ramo','id_ur','gpo_funcional','id_funcion','id_subfuncion','id_ai',
 'id_modalidad','id_pp','id_capitulo','id_concepto','id_partida_generica',
 'id_partida_especifica','id_tipogasto','id_ff','id_entidad_federativa',
 'id_clave_cartera']

duplicados = df.duplicated(subset=niveles).sum()
print(f"Registros duplicados por combinación clave: {duplicados}")
df_sin_niveles = df.drop(columns=niveles)
df_sin_niveles[df.duplicated(subset=niveles, keep=False)]

Registros duplicados por combinación clave: 2


,ciclo,desc_ramo,desc_ur,desc_gpo_funcional,desc_funcion,desc_subfuncion,desc_ai,desc_modalidad,desc_pp,desc_capitulo,...,desc_partida_especifica,desc_tipogasto,desc_ff,entidad_federativa,monto_aprobado,monto_modificado,monto_aprobado_mensual,monto_modificado_mensual,monto_pagado,tipo_gasto
22929,2026.0,Instituto de Seguridad y Servicios Sociales de los Trabajadores del Estado,Instituto de Seguridad y Servicios Sociales de los Trabajadores del Estado,Desarrollo Social,Protección Social,Otros de Seguridad Social y Asistencia Social,Servicios de apoyo administrativo,Apoyo para el desarrollo de las funciones de gobierno,Actividades de apoyo Administrativo,Servicios generales,...,"Enteros de los seguros de cesantía y vejéz, invalidez y vida y riesgos de trabajo",Gasto corriente,Ingresos Propios,Ciudad de México,2.033869e+10,0.000000e+00,4.213228e+09,0.000000e+00,3.475451e+09,PROGRAMABLE
22938,2026.0,Instituto de Seguridad y Servicios Sociales de los Trabajadores del Estado,Instituto de Seguridad y Servicios Sociales de los Trabajadores del Estado,Desarrollo Social,Protección Social,Otros de Seguridad Social y Asistencia Social,Servicios de apoyo administrativo,Apoyo para el desarrollo de las funciones de gobierno,Actividades de apoyo Administrativo,Servicios generales,...,Otros impuestos y derechos,Gasto corriente,Ingresos Propios,Ciudad de México,0.000000e+00,2.034273e+10,0.000000e+00,3.992751e+09,3.516563e+08,PROGRAMABLE
48361,2026.0,Hacienda y Crédito Público,Subtesorería de Operación,Gobierno,Asuntos Financieros y Hacendarios,Asuntos Hacendarios,Funciones de tesorería eficientes y transparentes,Prestación de Servicios Públicos,Servicios de administración de los recursos y valores federales,Servicios generales,...,Servicios bancarios y financieros,Gasto corriente,Recursos fiscales,Ciudad de México,3.965427e+08,2.621275e+08,4.900000e+07,6.521153e+07,1.204400e+07,PROGRAMABLE
48373,2026.0,Hacienda y Crédito Público,Subtesorería de Operación,Gobierno,Asuntos Financieros y Hacendarios,Asuntos Hacendarios,Funciones de tesorería eficientes y transparentes,Prestación de Servicios Públicos,Servicios de administración de los recursos y valores federales,Servicios generales,...,Servicios bancarios y financieros,Gasto corriente,Recursos fiscales,Ciudad de México,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.716611e+07,PROGRAMABLE


In [50]:
agrupacion = ['id_capitulo', 'desc_capitulo', 
                       'id_concepto', 'desc_concepto', 
                       'id_partida_generica', 'desc_partida_generica',
                       'id_partida_especifica', 'desc_partida_especifica']


duplicados_agrupacion = df[agrupacion].drop_duplicates(keep=False).dropna().reset_index(drop=True)

base = duplicados_agrupacion[['id_capitulo','id_concepto','id_partida_generica','id_partida_especifica']]
base.duplicated().sum()

PK_obj_gasto1 = (base['id_capitulo'].astype(int).astype(str) + 
                    base['id_concepto'].astype(int).astype(str) + 
                    base['id_partida_generica'].astype(int).astype(str) + 
                    base['id_partida_especifica'].astype(int).astype(str))

base.insert(0, 'PK_obj_gasto', PK_obj_gasto1)
base.duplicated(['PK_obj_gasto']).sum()





np.int64(0)

Se tienen valores repetidos, 2 renglones contienen la misma informacion salvo que difieren en la columna desc_partida_especifica en el primer caso, mientas que los otros dos renglones tienen las misma informacion salvo que difieren en los montos. Para no eliminar informacion  y como el dataset corresponde a un único ejercicio presupuestario, se consolidaron los registros duplicados sumando los montos financieros asociados a la misma clave presupuestaria.

In [51]:
columnas_monto = ['monto_aprobado', 'monto_aprobado_mensual', 'monto_modificado_mensual',
                   'monto_pagado', 'monto_modificado']
suma_22929_22938 = df.loc[[22929, 22938], columnas_monto].sum()
suma_48361_48373 = df.loc[[48361, 48373], columnas_monto].sum()
dfC = df.copy()
dfC.loc[22929, columnas_monto] = suma_22929_22938
dfC.loc[48361, columnas_monto] = suma_48361_48373
dfC = dfC.drop([22938, 48373]).reset_index(drop=True)
print(dfC.duplicated(subset=niveles).sum() == 0)

True


Bien, tenemos una base sin duplicados. Vamos a revisar algunos estadisticos de orden. 




In [52]:
display(dfC[columnas_monto].describe())

,monto_aprobado,monto_aprobado_mensual,monto_modificado_mensual,monto_pagado,monto_modificado
count,1.675910e+05,1.675910e+05,1.675910e+05,1.675910e+05,1.675910e+05
mean,7.009205e+07,1.878937e+07,1.913222e+07,1.726172e+07,7.013081e+07
std,3.817320e+09,1.062781e+09,1.035746e+09,9.845348e+08,3.790188e+09
min,-1.262015e+10,-1.936101e+09,-1.936101e+09,-1.115696e+10,-1.262015e+10
25%,3.920000e+02,0.000000e+00,0.000000e+00,0.000000e+00,1.102135e+04
50%,7.122400e+04,6.251000e+03,1.200000e+04,3.306000e+03,1.302960e+05
75%,9.311270e+05,1.523205e+05,2.044970e+05,1.190972e+05,1.279684e+06
max,1.002989e+12,2.525106e+11,2.558164e+11,2.552489e+11,1.002866e+12


No hay problema en los valores negativos, estos pueden representar un cargo y con respecto a las demas estadisticas como solo realizaremos un resumen de informcion, no tiene caso analizarlas. Lo que sigue es checar que la informacion sea consistente, al menos hablando de los montos.

In [53]:
print(f"Renglones donde mensual > anual (aprobado): {(dfC["monto_aprobado_mensual"] > dfC["monto_aprobado"]).sum()}")
dfC[dfC["monto_aprobado_mensual"] > dfC["monto_aprobado"]]

Renglones donde mensual > anual (aprobado): 1


,ciclo,id_ramo,desc_ramo,id_ur,desc_ur,gpo_funcional,desc_gpo_funcional,id_funcion,desc_funcion,id_subfuncion,...,desc_ff,id_entidad_federativa,entidad_federativa,id_clave_cartera,monto_aprobado,monto_modificado,monto_aprobado_mensual,monto_modificado_mensual,monto_pagado,tipo_gasto
3016,2026.0,50.0,Instituto Mexicano del Seguro Social,GYR,Instituto Mexicano del Seguro Social,2.0,Desarrollo Social,3.0,Salud,2.0,...,Ingresos Propios,9.0,Ciudad de México,0,-1.262015e+10,-1.262015e+10,-1.936101e+09,-1.936101e+09,-1.886093e+09,PROGRAMABLE


El termino negativo se debe a que es la forma en que se registran erogaciones por cuenta de terceros y devoluciones y el que sea mauno mayor a otro se debe al orden, esto no representa problema.

In [54]:
print(f"Renglones donde mensual > anual (modificado): {(dfC["monto_modificado_mensual"] > dfC["monto_modificado"]).sum()}")
dfC[dfC["monto_modificado_mensual"] > dfC["monto_modificado"]].head()

Renglones donde mensual > anual (modificado): 23


,ciclo,id_ramo,desc_ramo,id_ur,desc_ur,gpo_funcional,desc_gpo_funcional,id_funcion,desc_funcion,id_subfuncion,...,desc_ff,id_entidad_federativa,entidad_federativa,id_clave_cartera,monto_aprobado,monto_modificado,monto_aprobado_mensual,monto_modificado_mensual,monto_pagado,tipo_gasto
3016,2026.0,50.0,Instituto Mexicano del Seguro Social,GYR,Instituto Mexicano del Seguro Social,2.0,Desarrollo Social,3.0,Salud,2.0,...,Ingresos Propios,9.0,Ciudad de México,0,-1.262015e+10,-1.262015e+10,-1.936101e+09,-1.936101e+09,-1.886093e+09,PROGRAMABLE
26128,2026.0,52.0,Petróleos Mexicanos,TYY,Pemex Consolidado,1.0,Gobierno,3.0,Coordinación de la Política de Gobierno,4.0,...,Ingresos Propios,20.0,Oaxaca,0,8.728700e+04,-1.894500e+04,2.018400e+04,0.000000e+00,0.000000e+00,PROGRAMABLE
26138,2026.0,52.0,Petróleos Mexicanos,TYY,Pemex Consolidado,1.0,Gobierno,3.0,Coordinación de la Política de Gobierno,4.0,...,Ingresos Propios,20.0,Oaxaca,0,8.209300e+04,-1.284600e+04,0.000000e+00,0.000000e+00,0.000000e+00,PROGRAMABLE
26139,2026.0,52.0,Petróleos Mexicanos,TYY,Pemex Consolidado,1.0,Gobierno,3.0,Coordinación de la Política de Gobierno,4.0,...,Ingresos Propios,27.0,Tabasco,0,1.844274e+06,-4.467600e+04,0.000000e+00,0.000000e+00,0.000000e+00,PROGRAMABLE
26189,2026.0,52.0,Petróleos Mexicanos,TYY,Pemex Consolidado,1.0,Gobierno,3.0,Coordinación de la Política de Gobierno,4.0,...,Ingresos Propios,20.0,Oaxaca,0,2.992380e+05,-1.668830e+05,6.947700e+04,0.000000e+00,0.000000e+00,PROGRAMABLE


## Modelo estrella

Esto se justifica porque el mensual refleja ajustes específicos de un periodo, mientras que el modificado anual refleja el saldo acumulado neto. Ahora lo que se procese 

Vamos a tomar la tabla resultante y la particionaremos en subtablas para construir un modelo estrella y poder visualizar en Power Bi. 

In [55]:
# Generación de Tablas de Dimensiones

output_dir = r"Tablas_parquet"
os.makedirs(output_dir, exist_ok=True)

dim_tgasto = dfC[['tipo_gasto']].drop_duplicates().dropna().reset_index(drop=True)
dim_ramo = dfC[['id_ramo', 'desc_ramo']].drop_duplicates().dropna().reset_index(drop=True)
dim_ur = dfC[['id_ur','desc_ur']].drop_duplicates().dropna().reset_index(drop=True)

dim_func = dfC[['gpo_funcional', 'desc_gpo_funcional', 
                'id_funcion','desc_funcion', 
                'id_subfuncion', 'desc_subfuncion']].drop_duplicates().dropna().reset_index(drop=True)

PK_fun = (dim_func['gpo_funcional'].astype(int).astype(str)+ 
         dim_func['id_funcion'].astype(int).astype(str)+
         dim_func['id_subfuncion'].astype(int).astype(str))

dim_func.insert(0, 'PK_func', PK_fun)

dim_programa = dfC[['id_modalidad', 'desc_modalidad',
                    'id_pp',  'desc_pp']].drop_duplicates().dropna().reset_index(drop=True)

PK_prog = (dim_programa['id_modalidad'].astype(str) +  dim_programa['id_pp'].astype(int).astype(str))
dim_programa.insert(0, 'PK_prog', PK_prog)

dim_objeto_gasto = dfC[['id_capitulo', 'desc_capitulo', 
                       'id_concepto', 'desc_concepto', 
                       'id_partida_generica', 'desc_partida_generica',
                       'id_partida_especifica', 'desc_partida_especifica']].drop_duplicates().dropna().reset_index(drop=True)

PK_obj_gasto = (dim_objeto_gasto['id_capitulo'].astype(int).astype(str) + 
                    dim_objeto_gasto['id_concepto'].astype(int).astype(str) + 
                    dim_objeto_gasto['id_partida_generica'].astype(int).astype(str) + 
                    dim_objeto_gasto['id_partida_especifica'].astype(int).astype(str))

dim_objeto_gasto.insert(0, 'PK_obj_gasto', PK_obj_gasto)

dim_ff = dfC[['id_ff', 'desc_ff']].drop_duplicates().dropna().reset_index(drop=True)
dim_entidad = dfC[['id_entidad_federativa', 'entidad_federativa']].drop_duplicates().dropna().reset_index(drop=True)

#Tabla de Hechos
columnas_hechos = [
    'ciclo', 'id_ramo', 'id_ur', 'gpo_funcional', 'id_funcion', 'id_subfuncion',
    'id_modalidad', 'id_pp', 'id_capitulo', 'id_concepto', 'id_partida_generica',
    'id_partida_especifica', 'id_tipogasto', 'id_ff', 'id_entidad_federativa',
    'monto_aprobado', 'monto_modificado','monto_aprobado_mensual',
    'monto_modificado_mensual', 'monto_pagado', 'tipo_gasto']


fact_gasto = dfC[columnas_hechos].copy()

fact_gasto['FK_func'] = (dfC['gpo_funcional'].astype(int).astype(str) + 
                         dfC['id_funcion'].astype(int).astype(str) +
                         dfC['id_subfuncion'].astype(int).astype(str))

fact_gasto['FK_prog'] = (dfC['id_modalidad'].astype(str) +
                            dfC['id_pp'].astype(int).astype(str))

fact_gasto['FK_obj_gasto'] = (dfC['id_capitulo'].astype(int).astype(str) + 
                              dfC['id_concepto'].astype(int).astype(str) + 
                              dfC['id_partida_generica'].astype(int).astype(str) + 
                              dfC['id_partida_especifica'].astype(int).astype(str)) 

# Exportar Dimensiones
dim_tgasto.to_parquet(f"{output_dir}/dim_tgasto.parquet", index=False, engine='pyarrow')
dim_ramo.to_parquet(f"{output_dir}/dim_ramo.parquet", index=False, engine='pyarrow')
dim_ur.to_parquet(f"{output_dir}/dim_ur.parquet", index=False, engine='pyarrow')
dim_func.to_parquet(f"{output_dir}/dim_func.parquet", index=False, engine='pyarrow')
dim_programa.to_parquet(f"{output_dir}/dim_programa.parquet", index=False, engine='pyarrow')
dim_objeto_gasto.to_parquet(f"{output_dir}/dim_objeto_gasto.parquet", index=False, engine='pyarrow')
dim_ff.to_parquet(f"{output_dir}/dim_ff.parquet", index=False, engine='pyarrow')
dim_entidad.to_parquet(f"{output_dir}/dim_entidad.parquet", index=False, engine='pyarrow')
fact_gasto.to_parquet(f"{output_dir}/fact_gasto.parquet", index=False, engine='pyarrow')

In [1]:
# ebncuentra los valores duplicados en la tabla dim_objeto_gasto 
duplicates = dim_objeto_gasto[dim_objeto_gasto.duplicated(keep=False)]
duplicates
# '3000390039239202' busca ese valor en la tabla dim_objeto_gasto para ver si hay duplicados
dim_objeto_gasto[dim_objeto_gasto['PK_obj_gasto'] == '3000390039239202']

NameError: name 'dim_objeto_gasto' is not defined

In [ ]:
dim_objeto_gasto = dfC[['id_capitulo', 'desc_capitulo', 
                       'id_concepto', 'desc_concepto', 
                       'id_partida_generica', 'desc_partida_generica',
                       'id_partida_especifica', 'desc_partida_especifica']].drop_duplicates().dropna().reset_index(drop=True)

PK_obj_gasto = (dim_objeto_gasto['id_capitulo'].astype(int).astype(str) + 
                    dim_objeto_gasto['id_concepto'].astype(int).astype(str) + 
                    dim_objeto_gasto['id_partida_generica'].astype(int).astype(str) + 
                    dim_objeto_gasto['id_partida_especifica'].astype(int).astype(str))

dim_objeto_gasto.insert(0, 'PK_obj_gasto', PK_obj_gasto)

dim_objeto_gasto.diplicated(['PK_obj_gasto']).sum()

,id_capitulo,desc_capitulo,id_concepto,desc_concepto,id_partida_generica,desc_partida_generica,id_partida_especifica,desc_partida_especifica
0,1000.0,Servicios personales,1100.0,Remuneraciones al personal de carácter permanente,113.0,Sueldos base al personal permanente,11301.0,Sueldos base
1,1000.0,Servicios personales,1300.0,Remuneraciones adicionales y especiales,131.0,Primas por años de servicios efectivos prestados,13101.0,Prima quinquenal por años de servicios efectivos prestados
2,1000.0,Servicios personales,1300.0,Remuneraciones adicionales y especiales,132.0,"Primas de vacaciones, dominical y gratificación de fin de año",13201.0,Primas de vacaciones y dominical
3,1000.0,Servicios personales,1300.0,Remuneraciones adicionales y especiales,132.0,"Primas de vacaciones, dominical y gratificación de fin de año",13202.0,Aguinaldo o gratificación de fin de año
4,1000.0,Servicios personales,1300.0,Remuneraciones adicionales y especiales,133.0,Horas extraordinarias,13301.0,Remuneraciones por horas extraordinarias
...,...,...,...,...,...,...,...,...
441,2000.0,Materiales y suministros,2200.0,Alimentos y utensilios,221.0,Productos alimenticios para personas,22199.0,Alimentos y utensilios
442,5000.0,"Bienes muebles, inmuebles e intangibles",5100.0,Mobiliario y equipo de administración,511.0,Muebles de oficina y estantería,51199.0,Mobiliario y equipo de administración
443,3000.0,Servicios generales,3300.0,"Servicios profesionales, científicos, técnicos y otros servicios",339.0,"Servicios profesionales, científicos y técnicos integrales",33905.0,Servicios integrales en materia de seguridad pública y nacional
444,4000.0,"Transferencias, asignaciones, subsidios y otras ayudas",4800.0,Donativos,485.0,Donativos internacionales,48501.0,Donativos internacionales


In [32]:
dim_objeto_gasto[dim_objeto_gasto.duplicated(subset=['PK_obj_gasto'], keep=False)]

,PK_obj_gasto,id_capitulo,desc_capitulo,id_concepto,desc_concepto,id_partida_generica,desc_partida_generica,id_partida_especifica,desc_partida_especifica
26,2000290029329301,2000.0,Materiales y suministros,2900.0,"Herramientas, refacciones y accesorios menores",293.0,"Refacciones y accesorios menores de mobiliario y equipo de administración, educacional y recreativo",29301.0,"Refacciones y accesorios menores de mobiliario y equipo de administración,educacional y recreativo"
27,2000290029429401,2000.0,Materiales y suministros,2900.0,"Herramientas, refacciones y accesorios menores",294.0,Refacciones y accesorios menores de equipo de cómputo y tecnologías de la información,29401.0,Refacciones y accesorios para equipo de cómputo y telecomunicaciones
32,3000310031431401,3000.0,Servicios generales,3100.0,Servicios básicos,314.0,Telefonía tradicional,31401.0,Servicio telefónco convencional
49,3000390039239202,3000.0,Servicios generales,3900.0,Otros servicios generales,392.0,Impuestos y derechos,39202.0,Otros impuestos y derechos
51,1000120012212201,1000.0,Servicios personales,1200.0,Remuneraciones al personal de carácter transitorio,122.0,Sueldos base al personal eventual,12201.0,Remuneraciones al personal eventual
55,2000210021421401,2000.0,Materiales y suministros,2100.0,"Materiales de administración, emisión de documentos y artículos oficiales",214.0,"Materiales, útiles y equipos menores de tecnologías de la información y comunicaciones",21401.0,Materiales y útiles consumibles para el procesamiento en equipos y bienes informáticos
75,3000310031631603,3000.0,Servicios generales,3100.0,Servicios básicos,316.0,Servicios de telecomunicaciones y satélites,31603.0,Servicios de internet
86,3000320032732701,3000.0,Servicios generales,3200.0,Servicios de arrendamiento,327.0,Arrendamiento de activos intangibles,32701.0,"Patentes, derechos de autor, regalías y otros"
89,3000330033333301,3000.0,Servicios generales,3300.0,"Servicios profesionales, científicos, técnicos y otros servicios",333.0,"Servicios de consultoría administrativa, procesos, técnica y en tecnologías de la información",33301.0,Servicios de desarrollo de aplicaciones informáticas
94,3000330033633605,3000.0,Servicios generales,3300.0,"Servicios profesionales, científicos, técnicos y otros servicios",336.0,"Servicios de apoyo administrativo, traducción, fotocopiado e impresión",33605.0,Información en medios masivos derivada de la operación y administración de las dependencias y entidades
